In [1]:
import pandas as pd
import os
import sys
import json
from pathlib import Path
import numpy as np

# Notebook is in notebooks/, so repo root is parent
REPO_ROOT = Path.cwd().parent
SRC_PATH = REPO_ROOT / "src"

# Insert src at the front of sys.path so imports work
sys.path.insert(0, str(SRC_PATH))

In [2]:
import importlib
import preprocessing.vitals_preprocessing as vp

importlib.reload(vp)

<module 'preprocessing.vitals_preprocessing' from '/Users/brandonng/Documents/GitHub/ClinicalDigitalTwin/src/preprocessing/vitals_preprocessing.py'>

In [3]:
# Get repo root relative to the current notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Load static preprocessing config
config_path = os.path.join(repo_root, "configs", "vitals_preprocessing_params.json")
with open(config_path, "r") as f:
    config = json.load(f)

# Set input and output directories
in_dir = os.path.join(repo_root, config["paths"]["in_dir"])

In [21]:
from preprocessing.vitals_preprocessing import preprocess_lab_events

labevents_vitals = preprocess_lab_events(in_dir, config)
print(labevents_vitals.shape)
labevents_vitals.head(5)

(230070, 6)


label,subject_id,hadm_id,charttime_lab,Lab Oxygen Saturation %,Lab Temperature (C),chartdate
0,10001843,26133978.0,2134-12-05 19:30:00,NaN,36.6,2134-12-05
1,10001843,26133978.0,2134-12-05 21:29:00,81.0,NaN,2134-12-05
2,10001884,26184834.0,2131-01-11 06:37:00,81.0,36.1,2131-01-11
3,10001884,26184834.0,2131-01-11 11:33:00,72.0,NaN,2131-01-11
4,10001884,26184834.0,2131-01-11 22:21:00,NaN,36.9,2131-01-11


In [ ]:
from preprocessing.vitals_preprocessing import preprocess_ed_vitals

ed_vitals = preprocess_ed_vitals(in_dir, config)
print(ed_vitals.shape)
ed_vitals.head(5)


(1225239, 10)


,subject_id,stay_id,charttime_ed,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp,chartdate
0,17195991,38649090,2110-01-11 21:45:00,NaN,141.0,24.0,94.0,NaN,NaN,2110-01-11
1,17195991,38649090,2110-01-11 21:50:00,NaN,123.0,24.0,91.0,151.0,120.0,2110-01-11
2,17195991,38649090,2110-01-11 22:00:00,NaN,133.0,23.0,99.0,180.0,86.0,2110-01-11
3,17195991,38649090,2110-01-11 22:07:00,NaN,164.0,24.0,99.0,198.0,116.0,2110-01-11
4,17195991,38649090,2110-01-11 22:23:00,NaN,130.0,16.0,100.0,235.0,126.0,2110-01-11


In [8]:
print(ed_vitals.isna().mean() * 100)

subject_id                 0.000000
stay_id                    0.000000
charttime_ed               0.000000
ED Temperature (F)        39.603294
ED Heart Rate              4.283572
ED Respitory Rate          5.678402
ED Oxygen Saturation %     8.752986
ED sbp                     5.075581
ED dbp                     5.075581
chartdate                  0.000000
dtype: float64


In [19]:
from preprocessing.vitals_preprocessing import preprocess_omr

omr_data = preprocess_omr(in_dir, config)
print(omr_data.shape)
omr_data.head(5)

(2229541, 6)


result_name,subject_id,chartdate,Hosp BMI (kg/m2),Hosp Height (Inches),Hosp Weight (Lbs),Hosp Blood Pressure
0,10000032,2180-04-27,None,None,None,110/65
1,10000032,2180-05-07,None,None,None,None
2,10000032,2180-05-25,None,None,None,106/60
3,10000032,2180-06-01,None,None,None,121/77
4,10000032,2180-06-22,None,None,None,100/60


In [ ]:
# First, make sure time columns exist
omr_data['chartdate'] = pd.to_datetime(omr_data['chartdate'])
ed_vitals['charttime'] = pd.to_datetime(ed_vitals['charttime'])
ed_vitals['chartdate'] = ed_vitals['charttime'].dt.normalize()
labevents_vitals['charttime'] = pd.to_datetime(labevents_vitals['charttime'])
labevents_vitals['chartdate'] = labevents_vitals['charttime'].dt.normalize()

# Rename for clarity
ed_vitals = ed_vitals.rename(columns={'charttime': 'charttime_ed'})
labevents_vitals = labevents_vitals.rename(columns={'charttime': 'charttime_lab'})

ed_omr = ed_vitals.merge(
    omr_data,
    on=['subject_id', 'chartdate'],
    how='right'
)

final = labevents_vitals.merge(
    ed_omr,
    on=['subject_id', 'chartdate'],
    how='right'
)

# Now sorting works
final = final.sort_values(['subject_id', 'chartdate', 'charttime_lab', 'charttime_ed'])
final

,subject_id,hadm_id,charttime_lab,Lab Oxygen Saturation %,Lab Temperature (C),chartdate,stay_id,charttime_ed,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp,Hosp BMI (kg/m2),Hosp Height (Inches),Hosp Weight (Lbs),Hosp Blood Pressure
0,10000032,NaN,NaT,NaN,NaN,2180-04-27,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,110/65
1,10000032,NaN,NaT,NaN,NaN,2180-05-07,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
2,10000032,NaN,NaT,NaN,NaN,2180-05-25,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,106/60
3,10000032,NaN,NaT,NaN,NaN,2180-06-01,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,121/77
4,10000032,NaN,NaT,NaN,NaN,2180-06-22,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,100/60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2415173,19999828,NaN,NaT,NaN,NaN,2148-02-26,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,115/79
2415174,19999828,NaN,NaT,NaN,NaN,2148-04-29,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,105/67
2415175,19999828,NaN,NaT,NaN,NaN,2148-07-22,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,104/76
2415176,19999828,NaN,NaT,NaN,NaN,2148-10-19,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,112/73


In [20]:
from preprocessing.vitals_preprocessing import merge_vitals

vitals_data = merge_vitals(omr_data, ed_vitals ,labevents_vitals)
print(vitals_data.shape)
vitals_data.head(5)

(2415178, 18)


,subject_id,hadm_id,charttime_lab,Lab Oxygen Saturation %,Lab Temperature (C),chartdate,stay_id,charttime_ed,ED Temperature (F),ED Heart Rate,ED Respitory Rate,ED Oxygen Saturation %,ED sbp,ED dbp,Hosp BMI (kg/m2),Hosp Height (Inches),Hosp Weight (Lbs),Hosp Blood Pressure
0,10000032,NaN,NaT,NaN,NaN,2180-04-27,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,110/65
1,10000032,NaN,NaT,NaN,NaN,2180-05-07,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
2,10000032,NaN,NaT,NaN,NaN,2180-05-25,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,106/60
3,10000032,NaN,NaT,NaN,NaN,2180-06-01,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,121/77
4,10000032,NaN,NaT,NaN,NaN,2180-06-22,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,100/60
